# Import libraries

In [13]:
import glob
import pandas as pd
from pathlib import Path

---

# Read in the provided excel files

In [14]:
# Create a function to read in the files for code repeatability
def read_csv_files(folder_path, pattern='*.csv'):
    '''
        This function reads in all the files in a folder based on pattern matching and returns a dictionary of dataframes.
        It reads in .csv files by default. 
    '''
    # Path of the folder containing files.
    folder_path=Path(folder_path)
    
    # Get a list of all files in the folder
    files = folder_path.glob(pattern)
    
    # Create a dictionary to house each resulting dataframe from the excel files
    dataframes = {}

    # Read in each excel file into a separate pandas dataframe
    for file in files:
        # Add dataframe to the dictionary
        dataframes[file.stem] = pd.read_csv(file) # The engine parameter might be optional depending on the device.
    
    return dataframes
    

In [15]:
RAW_DATA_DIR = Path("../data/raw")
dataframes = read_csv_files(RAW_DATA_DIR)
dataframes.keys()

dict_keys(['medication', 'patient', 'restock', 'nurse', 'transaction_type', 'machine', 'transaction', 'hospital', 'data_dictionary', 'hospital_ward'])

---

# Display data dictionary

In [16]:
dataframes['data_dictionary']

,table_name,table_description,column,column_description
0,hospital,Hospital site transactional information.,technical_id,internal table ID
1,hospital,Hospital site transactional information.,hospital_id,internal hospital ID
2,hospital,Hospital site transactional information.,hospital_abbreviation,abbreviated hospital ID name
3,hospital,Hospital site transactional information.,hospital_name,Name of hospital
4,hospital_ward,Ward area transactional information for each h...,technical_id,internal table ID
5,hospital_ward,Ward area transactional information for each h...,hospital_ward_id,internal hospital ward ID
6,hospital_ward,Ward area transactional information for each h...,hospital_ward_name,hospital ward name
7,hospital_ward,Ward area transactional information for each h...,hospital_abbreviation,abbreviated hospital ID name
8,machine,Machine (ADC) transactional information,technical_id,internal table ID
9,machine,Machine (ADC) transactional information,machine_id,internal machine ID


---

# Explore each dataframe, determine relevance to user story, and clean if necessary

In [17]:
dataframes.keys()

dict_keys(['medication', 'patient', 'restock', 'nurse', 'transaction_type', 'machine', 'transaction', 'hospital', 'data_dictionary', 'hospital_ward'])

### The machine table

In [18]:
machine = dataframes['machine']
machine.head()

,technical_id,machine_id,machine_identifier,machine_name,machine_type,hospital_abbreviation
0,1,4,SVJOINT1,JOINT1 (ORTHO),CTS,SV
1,2,6,SV6W,6W,CTS,SV
2,3,12,SVOR3,OR 3,CTS,SV
3,4,16,SVONCOLOGY,Oncology,CTS,SV
4,5,19,SVCICU2,CICU2,CTS,SV


### Information in this table is irrelevant to the user story

---

### The medication table

In [19]:
medication = dataframes['medication']
medication.head()

,technical_id,machine_identifier,medication_id,medication_detail_id,medication,medication_unit,medication_unit_cost
0,1,SVJOINT1,969384,34505,HYDROcodone/acetamino 1ea tablet,TAB,0.0
1,2,SVANES,969403,PRO200E,propofol (D 10mg/1ml 20ml ampule,AMP,0.0
2,3,SVANES,969407,MID1I,midazolam (VERS 2mg/2ml 2ml vial,VIAL,0.0
3,4,SVANES,969408,MID1I,midazolam (VERS 2mg/2ml 2ml vial,VIAL,0.0
4,5,SV7E,969434,OXY5TI,oxyCODONE IR (ROXICOD 5mg tablet,TAB,0.0


### The 'medication_id' and 'medication' columns are important for identifying the medication in each transaction

In [20]:
medication.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41023 entries, 0 to 41022
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   technical_id          41023 non-null  int64  
 1   machine_identifier    41020 non-null  object 
 2   medication_id         41023 non-null  int64  
 3   medication_detail_id  41023 non-null  object 
 4   medication            40976 non-null  object 
 5   medication_unit       39931 non-null  object 
 6   medication_unit_cost  41023 non-null  float64
dtypes: float64(1), int64(2), object(4)
memory usage: 2.2+ MB


In [21]:
# check if there are duplicated medication ids
medication['medication_id'].duplicated().sum()

np.int64(0)

In [22]:
# Determine the proportion of medications without medication name
medication['medication'].isnull().sum() / len(medication) * 100

np.float64(0.11456987543573117)

In [23]:
# Drop rows without medication name. 
# This should not be a problem considering that the entries affected is less than 1%
medication = medication.dropna(subset='medication')

In [24]:
medication.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40976 entries, 0 to 41022
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   technical_id          40976 non-null  int64  
 1   machine_identifier    40973 non-null  object 
 2   medication_id         40976 non-null  int64  
 3   medication_detail_id  40976 non-null  object 
 4   medication            40976 non-null  object 
 5   medication_unit       39929 non-null  object 
 6   medication_unit_cost  40976 non-null  float64
dtypes: float64(1), int64(2), object(4)
memory usage: 2.5+ MB


---

### The transaction type table

In [25]:
transaction_type = dataframes['transaction_type']
print(transaction_type.info())
transaction_type.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   technical_id             23 non-null     int64 
 1   transaction_type         23 non-null     object
 2   transaction_type_action  23 non-null     object
dtypes: int64(1), object(2)
memory usage: 684.0+ bytes
None


,technical_id,transaction_type,transaction_type_action
0,1,A,Inactive Access
1,2,B,Bedside
2,3,C,Cycle Count
3,4,D,Discrepancy
4,5,E,Emergency


In [26]:
transaction_type

,technical_id,transaction_type,transaction_type_action
0,1,A,Inactive Access
1,2,B,Bedside
2,3,C,Cycle Count
3,4,D,Discrepancy
4,5,E,Emergency
5,6,F,Dispensing Error
6,7,G,Pick
7,8,I,Issue
8,9,K,Destock
9,10,L,Receive


### This table is only relevant because logically, not all transaction types leading to a stock count of zero (0) can be considered as a stockout situation. For example, if you destock (transaction_type K) or Transfer (transaction_type T), that is an intentional attempt to remove the medication out of the dispensing cabinet. It doesn't mean the medication is out of stock.

---

### The nurse table

In [27]:
nurse = dataframes['nurse']
nurse.head()

,technical_id,nurse_id,user_type,user_name
0,1,2732,NURSE,MADALINE SHOVE
1,2,2824,ANESTHESIA,JOVAN PLUMP
2,3,4081,NURSE,AMY TREAGER
3,4,651763,NURSE,BETSEY LIPSIE
4,5,651849,NURSE,FRANCINE HOCHSTATTER


### Information in this table is irrelevant to the user story

---

### The hospital table

In [28]:
hospital = dataframes['hospital']
hospital.head()

,technical_id,hospital_id,hospital_abbreviation,hospital_name
0,1,2,SJ,St Joseph's
1,2,597549,NY,North York


#### Information about just two hospitals are present: Stamford-Vale and Stafford Tyne.

---

### The restock table

In [29]:
restock = dataframes['restock']
restock.head()

,technical_id,restock_id,restock_number,restock_route,destination_ward,destination_hospital
0,1,0,NaN,NaN,NaN,NaN
1,2,1390804,CPC01-0097740.00,Surgery Refill Mornin CP,NaN,NaN
2,3,1391470,CPC01-0097762.00,Surgery Refill Afterno CP,NaN,NaN
3,4,1391484,CPC01-0097761.00,Surgery Refill PM NARCS,NaN,NaN
4,5,1391660,CPC01-0097768.00,NaN,NaN,NaN


### Information in this table is irrelevant to the user story

---

### The transaction table

In [30]:
transaction = dataframes['transaction']
transaction.head()

,transaction_id,datetime,transaction_type,hospital_id,hospital_ward_id,machine_id,nurse_id,patient_id,medication_id,transaction_quantity,quantity_on_hand,restock_id
0,1,2022101800055550,V,2,43,646284,655065,0,997741,0.0,46,1508931
1,2,2025101800071300,R,2,62,100,654492,2207026,1025890,-2.0,9,0
2,3,2022101800071800,I,2,58,646282,654805,2203789,1022831,1.0,13,0
3,4,2022101800073000,I,2,4,70,654127,2206842,1036923,1.0,2,0
4,5,2022101800073200,I,2,82,45,655632,2207163,1038449,4.0,11,0


### The hospital_ward_id, medication_id, transaction_quantity, and quantity_on_hand columns in this table are extremely important to the user story

In [31]:
transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        1000 non-null   int64  
 1   datetime              1000 non-null   int64  
 2   transaction_type      1000 non-null   object 
 3   hospital_id           1000 non-null   int64  
 4   hospital_ward_id      1000 non-null   int64  
 5   machine_id            1000 non-null   int64  
 6   nurse_id              1000 non-null   int64  
 7   patient_id            1000 non-null   int64  
 8   medication_id         1000 non-null   int64  
 9   transaction_quantity  1000 non-null   float64
 10  quantity_on_hand      1000 non-null   int64  
 11  restock_id            1000 non-null   int64  
dtypes: float64(1), int64(10), object(1)
memory usage: 93.9+ KB


In [32]:
# Check for unique hospital ids to know the number of hospitals represented
transaction['hospital_id'].unique()

array([2])

#### Note: It seems all the transactions occurred in the hospital with hospital_id = 2. That means all transactions occured at the Stamford-Vale hospital.

---

### The hospital ward table

In [33]:
hospital_ward = dataframes['hospital_ward']
hospital_ward.head()

,technical_id,hospital_ward_id,hospital_ward_name,hospital_abbreviation
0,1,4,JOINT,SJ
1,2,5,6W,SJ
2,3,12,OR,SJ
3,4,15,ONCOLOGY,SJ
4,5,18,CICU12,SJ


In [39]:
hospital_ward = hospital_ward.drop(columns="technical_id")
hospital_ward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   hospital_ward_id       44 non-null     int64 
 1   hospital_ward_name     43 non-null     object
 2   hospital_abbreviation  43 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.2+ KB


In [40]:
# Check if each entry of hospital_ward_id is unique in this table
hospital_ward['hospital_ward_id'].duplicated().sum()

np.int64(0)

### The hospital_ward_id is necessary to know which ward a particular transaction occurs in

### The patient table

In [41]:
patient = dataframes['patient']
patient.head()

,technical_id,patient_id,patient_identifier_number,patient_type
0,1,0,NaN,NaN
1,2,1986300,3.725978e+09,INP
2,3,2017288,9.361677e+09,INP
3,4,2036474,4.343004e+08,INP
4,5,2061316,9.313283e+09,INP


### Information in this table is irrelevant to the user story 

---

# Create your final table for data exploration

### From the individual table examinations above, the tables and columns relevant to the user story are the following:
| Table |   | Columns |
|----------|---|----------|
|transaction |   | transaction_id, datetime, transaction_type, hospital_id, hospital_ward_id, medication_id, transaction_quantity, quantity_on_hand |
| transaction_type |   | transaction_type, transaction_type_action | 
| medication |   | medication_id, medication | 
| hospital |   | hospital_id, hospital_name |
| hospital_ward |   | hospital_ward_id, hospital_ward_name |


### Create a function to join the necessary tables one after the other to avoid confusion. The most important table is the transaction table. That will be the base table. 

In [42]:
# Create the funtion 'table_merge' that will perform the joins
def merge_tables():
    '''
        This function merges the transaction, transaction_type, medication, hospital, and hospital_ward tables. 
        It performs a left join and will not work for any other table(s)
    '''
    
    # Perform joins one after the other
    df_first_join = pd.merge(transaction, transaction_type, on='transaction_type', how='left')
    df_second_join = pd.merge(df_first_join, medication, on='medication_id', how='left')
    df_third_join = pd.merge(df_second_join, hospital, on='hospital_id', how='left')
    df_fourth_join = pd.merge(df_third_join, hospital_ward, on='hospital_ward_id', how='left')
    
    # Create a list of desired columns from the identified relevant columns
    desired_columns = ['transaction_id', 'datetime', 'transaction_type', 'transaction_type_action', 
                       'transaction_quantity', 'quantity_on_hand', 'medication_id', 'medication',
                       'hospital_id', 'hospital_name', 'hospital_ward_id', 'hospital_ward_name'
                      ]

    # subset only desired columns and create the final dataframe name df
    df = df_fourth_join[desired_columns]
    
    return df

In [44]:
df = merge_tables()
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           1000 non-null   int64  
 1   datetime                 1000 non-null   int64  
 2   transaction_type         1000 non-null   object 
 3   transaction_type_action  842 non-null    object 
 4   transaction_quantity     1000 non-null   float64
 5   quantity_on_hand         1000 non-null   int64  
 6   medication_id            1000 non-null   int64  
 7   medication               1000 non-null   object 
 8   hospital_id              1000 non-null   int64  
 9   hospital_name            1000 non-null   object 
 10  hospital_ward_id         1000 non-null   int64  
 11  hospital_ward_name       1000 non-null   object 
dtypes: float64(1), int64(6), object(5)
memory usage: 93.9+ KB
None


,transaction_id,datetime,transaction_type,transaction_type_action,transaction_quantity,quantity_on_hand,medication_id,medication,hospital_id,hospital_name,hospital_ward_id,hospital_ward_name
0,1,2022101800055550,V,Event,0.0,46,997741,nadolol (CORGARD) 40mg tablet,2,St Joseph's,43,RX
1,2,2025101800071300,R,Return,-2.0,9,1025890,bisacodyl (DULCOLAX) 5mg tablet,2,St Joseph's,62,3P1
2,3,2022101800071800,I,Issue,1.0,13,1022831,meropenem Vial2Bag 500mg,2,St Joseph's,58,2NE
3,4,2022101800073000,I,Issue,1.0,2,1036923,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,4,JOINT
4,5,2022101800073200,I,Issue,4.0,11,1038449,oxyCODONE IR 5mg tablet,2,St Joseph's,82,7TN


#### There seems to be some missing values for the 'transaction_type_action' column. Explore this

In [45]:
df.loc[df['transaction_type_action'].isnull()]

,transaction_id,datetime,transaction_type,transaction_type_action,transaction_quantity,quantity_on_hand,medication_id,medication,hospital_id,hospital_name,hospital_ward_id,hospital_ward_name
30,31,2022101802082900,W,NaN,0.50,0,1036782,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,36,EMD
31,32,2022101802091000,W,NaN,1.00,0,1036782,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,36,EMD
33,34,2022101802091100,W,NaN,0.75,0,1020842,LORazepam (ATIV 2mg/1ml 1ml vial,2,St Joseph's,36,EMD
35,36,2022101802110900,W,NaN,1.00,0,1036948,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,4,JOINT
37,38,2022101802232340,G,NaN,1.00,26,1030894,fentaNYL/bupi 3mcg/1ml 250ml bag,2,St Joseph's,43,RX
...,...,...,...,...,...,...,...,...,...,...,...,...
987,988,2022101815160680,G,NaN,5.00,14,1014745,calcium gluc 100mg/1ml 10ml vial,2,St Joseph's,43,RX
990,991,2022101815181420,G,NaN,10.00,65,1013885,magnesium sulf 4meq/1ml 2ml vial,2,St Joseph's,43,RX
992,993,2022101815182760,G,NaN,25.00,1322,1033762,HYDROcodone/acetamino 1ea tablet,2,St Joseph's,43,RX
995,996,2022101815185540,G,NaN,10.00,22,1009079,sterile water 20mL vial,2,St Joseph's,43,RX


#### Knowing there are no missing values in the transaction_type table, let's check the transaction_type column of our final dataframe to identify what the issue is

In [46]:
df['transaction_type'].unique()

array(['V', 'R', 'I', 'K', 'S', 'W', 'N', 'G', 'C', 'U', 'D', 'X', ' W',
       'L', 'O', 'M', ' V', 'G ', 'V '], dtype=object)

#### Some values (e.g., V, W, G) seem duplicated with the difference being leading or trailing spaces. This would likely be the same in the original transaction table which is the base table and the transaction_type table

In [47]:
# Let's check the original transaction table which is our base table
transaction['transaction_type'].unique()

array(['V', 'R', 'I', 'K', 'S', 'W', 'N', 'G', 'C', 'U', 'D', 'X', ' W',
       'L', 'O', 'M', ' V', 'G ', 'V '], dtype=object)

In [48]:
# Let's also check the original transaction_type table
transaction_type['transaction_type'].unique()

array(['A', 'B', 'C', 'D', 'E', 'F', ' G', 'I', 'K', 'L', 'M', 'N', 'O',
       'P ', 'Q', 'R', 'S', 'T', 'U', 'V', 'W ', 'X', 'Z'], dtype=object)

#### This error has to be fixed from both tables

In [49]:
transaction['transaction_type'] = transaction['transaction_type'].str.strip()
transaction['transaction_type'].unique()

array(['V', 'R', 'I', 'K', 'S', 'W', 'N', 'G', 'C', 'U', 'D', 'X', 'L',
       'O', 'M'], dtype=object)

In [50]:
transaction_type['transaction_type'] = transaction_type['transaction_type'].str.strip()
transaction_type['transaction_type'].unique()

array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'K', 'L', 'M', 'N', 'O',
       'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Z'], dtype=object)

#### Now that the issue is sorted, let's merge the tables again

In [51]:
df = merge_tables()
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           1000 non-null   int64  
 1   datetime                 1000 non-null   int64  
 2   transaction_type         1000 non-null   object 
 3   transaction_type_action  1000 non-null   object 
 4   transaction_quantity     1000 non-null   float64
 5   quantity_on_hand         1000 non-null   int64  
 6   medication_id            1000 non-null   int64  
 7   medication               1000 non-null   object 
 8   hospital_id              1000 non-null   int64  
 9   hospital_name            1000 non-null   object 
 10  hospital_ward_id         1000 non-null   int64  
 11  hospital_ward_name       1000 non-null   object 
dtypes: float64(1), int64(6), object(5)
memory usage: 93.9+ KB
None


,transaction_id,datetime,transaction_type,transaction_type_action,transaction_quantity,quantity_on_hand,medication_id,medication,hospital_id,hospital_name,hospital_ward_id,hospital_ward_name
0,1,2022101800055550,V,Event,0.0,46,997741,nadolol (CORGARD) 40mg tablet,2,St Joseph's,43,RX
1,2,2025101800071300,R,Return,-2.0,9,1025890,bisacodyl (DULCOLAX) 5mg tablet,2,St Joseph's,62,3P1
2,3,2022101800071800,I,Issue,1.0,13,1022831,meropenem Vial2Bag 500mg,2,St Joseph's,58,2NE
3,4,2022101800073000,I,Issue,1.0,2,1036923,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,4,JOINT
4,5,2022101800073200,I,Issue,4.0,11,1038449,oxyCODONE IR 5mg tablet,2,St Joseph's,82,7TN


In [52]:
# Change the data type of the datetime column to datetime
df['datetime'] = pd.to_datetime(df['datetime'].astype('str').str[:14])
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   transaction_id           1000 non-null   int64         
 1   datetime                 1000 non-null   datetime64[ns]
 2   transaction_type         1000 non-null   object        
 3   transaction_type_action  1000 non-null   object        
 4   transaction_quantity     1000 non-null   float64       
 5   quantity_on_hand         1000 non-null   int64         
 6   medication_id            1000 non-null   int64         
 7   medication               1000 non-null   object        
 8   hospital_id              1000 non-null   int64         
 9   hospital_name            1000 non-null   object        
 10  hospital_ward_id         1000 non-null   int64         
 11  hospital_ward_name       1000 non-null   object        
dtypes: datetime64[ns](1), float64(1), i

,transaction_id,datetime,transaction_type,transaction_type_action,transaction_quantity,quantity_on_hand,medication_id,medication,hospital_id,hospital_name,hospital_ward_id,hospital_ward_name
0,1,2022-10-18 00:05:55,V,Event,0.0,46,997741,nadolol (CORGARD) 40mg tablet,2,St Joseph's,43,RX
1,2,2025-10-18 00:07:13,R,Return,-2.0,9,1025890,bisacodyl (DULCOLAX) 5mg tablet,2,St Joseph's,62,3P1
2,3,2022-10-18 00:07:18,I,Issue,1.0,13,1022831,meropenem Vial2Bag 500mg,2,St Joseph's,58,2NE
3,4,2022-10-18 00:07:30,I,Issue,1.0,2,1036923,fentaNYL 100mCg/2ml 2ml vial,2,St Joseph's,4,JOINT
4,5,2022-10-18 00:07:32,I,Issue,4.0,11,1038449,oxyCODONE IR 5mg tablet,2,St Joseph's,82,7TN


### Now we have a final table with no missing values and information relevant to our user story that can be saved.

In [54]:
df.to_csv(Path.cwd() / "processed-data" / "final_table.csv", index=False)